In [2]:
# ============================================================
# COMPLETE DATA VISUALIZATION FOR MAXIMUM SUBARRAY PROJECT
# ============================================================
# Run this entire cell in Google Colab
# Upload your results.csv when prompted
# All plots will be generated and downloaded as a zip file
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
from google.colab import files
import zipfile
import io

# ============================================================
# 1. UPLOAD AND LOAD DATA
# ============================================================
print("=" * 60)
print("STEP 1: Upload your results.csv file")
print("=" * 60)

# Fixed file upload
uploaded_file = files.upload()

# Get the filename
file_name = list(uploaded_file.keys())[0]

# Load the data
df = pd.read_csv(io.BytesIO(uploaded_file[file_name]))
print(f"\n✅ Loaded {len(df)} rows")
print(f"✅ Algorithms: {df['algorithm'].unique().tolist()}")
print(f"✅ Dataset types: {df['dataset_type'].unique().tolist()}")
print(f"✅ Input sizes: {sorted(df['n'].unique())}")

# Set style for professional plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Create output directory
os.makedirs('plots_folder', exist_ok=True)

# ============================================================
# 2. PLOT 1: All algorithms comparison (Random dataset)
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Generating plots...")
print("=" * 60)

random_df = df[df['dataset_type'] == 'random']

plt.figure(figsize=(12, 7))
for algo in random_df['algorithm'].unique():
    algo_data = random_df[random_df['algorithm'] == algo]
    plt.plot(algo_data['n'], algo_data['time_ms'], 'o-', linewidth=2, markersize=8, label=algo)
plt.xlabel('Input Size (n)', fontsize=12)
plt.ylabel('Time (ms)', fontsize=12)
plt.title('Maximum Subarray: All Algorithms on Random Data', fontsize=14)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.savefig('plots_folder/plot1_all_algorithms.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Plot 1: plot1_all_algorithms.png")

# ============================================================
# 3. PLOT 2: Log-log plot (shows complexity slopes)
# ============================================================
plt.figure(figsize=(12, 7))
colors = ['red', 'blue', 'green', 'orange', 'purple']
for i, algo in enumerate(random_df['algorithm'].unique()):
    algo_data = random_df[random_df['algorithm'] == algo]
    plt.loglog(algo_data['n'], algo_data['time_ms'], 'o-', linewidth=2, markersize=8, color=colors[i], label=algo)
# Add theoretical slope reference lines
n_vals = np.array([100, 1000, 10000])
plt.loglog(n_vals, 1e-6 * n_vals**3, 'k--', alpha=0.5, linewidth=1, label='O(n³) reference')
plt.loglog(n_vals, 1e-4 * n_vals**2, 'k:', alpha=0.5, linewidth=1, label='O(n²) reference')
plt.loglog(n_vals, 1e-2 * n_vals, 'k-.', alpha=0.5, linewidth=1, label='O(n) reference')
plt.xlabel('Input Size (n) - log scale', fontsize=12)
plt.ylabel('Time (ms) - log scale', fontsize=12)
plt.title('Log-Log Plot: Empirical vs Theoretical Complexity', fontsize=14)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.savefig('plots_folder/plot2_loglog_complexity.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Plot 2: plot2_loglog_complexity.png")

# ============================================================
# 4. PLOT 3: Speedup vs Brute Force (O(n³) baseline)
# ============================================================
plt.figure(figsize=(12, 7))
baseline = random_df[random_df['algorithm'] == 'Brute Force (All Subarrays)'][['n', 'time_ms']].rename(columns={'time_ms': 'baseline_ms'})
for algo in random_df['algorithm'].unique():
    if algo == 'Brute Force (All Subarrays)':
        continue
    algo_data = random_df[random_df['algorithm'] == algo][['n', 'time_ms']]
    merged = pd.merge(algo_data, baseline, on='n')
    merged['speedup'] = merged['baseline_ms'] / merged['time_ms']
    plt.plot(merged['n'], merged['speedup'], 'o-', linewidth=2, markersize=8, label=algo)
plt.xlabel('Input Size (n)', fontsize=12)
plt.ylabel('Speedup Factor (vs O(n³) Brute Force)', fontsize=12)
plt.title('Speedup Over Brute Force (Log Scale)', fontsize=14)
plt.yscale('log')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.savefig('plots_folder/plot3_speedup_vs_bruteforce.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Plot 3: plot3_speedup_vs_bruteforce.png")

# ============================================================
# 5. PLOT 4: Dataset type comparison (Kadane only)
# ============================================================
kadane_df = df[df["algorithm"] == "Kadane's Algorithm"]
plt.figure(figsize=(10, 6))
for dtype in kadane_df['dataset_type'].unique():
    dtype_data = kadane_df[kadane_df['dataset_type'] == dtype]
    plt.plot(dtype_data['n'], dtype_data['time_ms'], 'o-', linewidth=2, markersize=8, label=dtype)
plt.xlabel('Input Size (n)', fontsize=12)
plt.ylabel('Time (ms)', fontsize=12)
plt.title("Kadane's Algorithm: Performance Across Dataset Types", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('plots_folder/plot4_kadane_dataset_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Plot 4: plot4_kadane_dataset_comparison.png")

# ============================================================
# 6. PLOT 5: Bar chart for n=3000 (final snapshot)
# ============================================================
n3000_df = df[df['n'] == 3000]
plt.figure(figsize=(12, 6))
algorithms = n3000_df['algorithm'].unique()
dataset_types = n3000_df['dataset_type'].unique()
x = np.arange(len(algorithms))
width = 0.25
for i, dtype in enumerate(dataset_types):
    dtype_data = n3000_df[n3000_df['dataset_type'] == dtype]
    times = [dtype_data[dtype_data['algorithm'] == algo]['time_ms'].values[0] for algo in algorithms]
    plt.bar(x + i*width, times, width, label=dtype)
plt.xlabel('Algorithm', fontsize=12)
plt.ylabel('Time (ms)', fontsize=12)
plt.title('Performance Comparison at n = 3000', fontsize=14)
plt.xticks(x + width, [a.replace(' (All Subarrays)', '') for a in algorithms], rotation=15, ha='right')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.savefig('plots_folder/plot5_n3000_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Plot 5: plot5_n3000_comparison.png")

# ============================================================
# 7. PLOT 6: Heatmap (time vs algorithm vs size)
# ============================================================
pivot_df = random_df.pivot(index='n', columns='algorithm', values='time_ms')
plt.figure(figsize=(10, 8))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Time (ms)'})
plt.title('Runtime Heatmap: Algorithm vs Input Size (Random Data)', fontsize=14)
plt.ylabel('Input Size (n)')
plt.xlabel('Algorithm')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('plots_folder/plot6_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Plot 6: plot6_heatmap.png")

# ============================================================
# 8. GENERATE SUMMARY TABLE
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Summary Table")
print("=" * 60)

summary = random_df.groupby(['algorithm', 'n'])['time_ms'].mean().unstack()
summary = summary.round(4)
print("\nRuntime (ms) on Random Dataset:")
print(summary)

max_n = random_df['n'].max()
bruteforce_time = random_df[(random_df['algorithm'] == 'Brute Force (All Subarrays)') & (random_df['n'] == max_n)]['time_ms'].values[0]
kadane_time = random_df[(random_df["algorithm"] == "Kadane's Algorithm") & (random_df['n'] == max_n)]['time_ms'].values[0]
speedup = bruteforce_time / kadane_time
print(f"\n📊 KEY FINDING: At n={max_n}, Kadane is {speedup:.0f}x faster than Brute Force")

# ============================================================
# 9. CREATE MARKDOWN TABLE (copy to report)
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Markdown Table (Copy this to your report)")
print("=" * 60)
print("\n| n | Brute Force (ms) | Better Brute (ms) | Prefix Sum (ms) | Divide & Conquer (ms) | Kadane (ms) |")
print("|---|------------------|-------------------|-----------------|----------------------|-------------|")
for n in sorted(random_df['n'].unique()):
    row = random_df[random_df['n'] == n]
    bf = row[row['algorithm'] == 'Brute Force (All Subarrays)']['time_ms'].values[0]
    bb = row[row['algorithm'] == 'Better Brute Force']['time_ms'].values[0]
    ps = row[row['algorithm'] == 'Prefix Sum']['time_ms'].values[0]
    dc = row[row['algorithm'] == 'Divide & Conquer']['time_ms'].values[0]
    kd = row[row["algorithm"] == "Kadane's Algorithm"]['time_ms'].values[0]
    print(f"| {n} | {bf:.4f} | {bb:.4f} | {ps:.4f} | {dc:.4f} | {kd:.4f} |")

# ============================================================
# 10. CREATE VERIFICATION SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Verification Summary")
print("=" * 60)

# Check if all algorithms agree on max_sum per (n, dataset_type)
verification = df.groupby(['n', 'dataset_type'])['max_sum'].nunique().reset_index()
verification['all_agree'] = verification['max_sum'] == 1
if verification['all_agree'].all():
    print("✅ VERIFICATION PASSED: All algorithms produce identical max_sum values")
else:
    print("⚠️ WARNING: Some algorithms produced different max_sum values")
    print(verification[~verification['all_agree']])

# ============================================================
# 11. DOWNLOAD ALL PLOTS (FIXED)
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Downloading all plots")
print("=" * 60)

# Create zip file
with zipfile.ZipFile('all_plots.zip', 'w') as zipf:
    for root, dirs, files_in_dir in os.walk('plots_folder'):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file), file)

# Download - FIXED: use files.download correctly
from google.colab import files as colab_files
colab_files.download('all_plots.zip')
print("\n✅ Download complete! 'all_plots.zip' contains all 6 plots")
print("\n" + "=" * 60)
print("🎉 DONE! All plots and tables generated successfully")
print("=" * 60)

STEP 1: Upload your results.csv file


Saving results.csv to results (1).csv

✅ Loaded 75 rows
✅ Algorithms: ['Brute Force (All Subarrays)', 'Better Brute Force', 'Prefix Sum', 'Divide & Conquer', "Kadane's Algorithm"]
✅ Dataset types: ['random', 'negative', 'positive']
✅ Input sizes: [np.int64(100), np.int64(500), np.int64(1000), np.int64(2000), np.int64(3000)]

STEP 2: Generating plots...
✅ Plot 1: plot1_all_algorithms.png
✅ Plot 2: plot2_loglog_complexity.png
✅ Plot 3: plot3_speedup_vs_bruteforce.png
✅ Plot 4: plot4_kadane_dataset_comparison.png
✅ Plot 5: plot5_n3000_comparison.png
✅ Plot 6: plot6_heatmap.png

STEP 3: Summary Table

Runtime (ms) on Random Dataset:
n                              100      500       1000       2000        3000
algorithm                                                                    
Better Brute Force           0.0422   1.1261    3.7828    10.8360     31.7550
Brute Force (All Subarrays)  0.7492  86.4219  627.2828  3644.7458  13766.6189
Divide & Conquer             0.0295   0.0599    0.1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete! 'all_plots.zip' contains all 6 plots

🎉 DONE! All plots and tables generated successfully
